In [1]:
import pandas as pd
from pathlib import Path

In [2]:
# Parquet with fcc county residuals
parquet_path = Path("/home/eprashar_solutions_corelogic_com/network-idx/data/features/fcc/broadband_coverage/county_residuals/fcc_coverage_county_residuals_AK_02.parquet")
df = pd.read_parquet(parquet_path)
print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 24 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   county_geoid                30 non-null     str    
 1   state_fips                  30 non-null     str    
 2   county_total_units          30 non-null     int64  
 3   places_total_units          30 non-null     int64  
 4   residual_units              30 non-null     int64  
 5   place_count                 30 non-null     int64  
 6   copper_speed_02_02_only     26 non-null     Float64
 7   copper_speed_10_1_only      26 non-null     Float64
 8   copper_speed_25_3_only      26 non-null     Float64
 9   copper_speed_100_20_only    26 non-null     Float64
 10  copper_speed_250_25_only    26 non-null     Float64
 11  copper_speed_1000_100_only  26 non-null     Float64
 12  cable_speed_02_02_only      26 non-null     Float64
 13  cable_speed_10_1_only       26 non-null     Floa

In [7]:
# Testing the county residuals script
from pathlib import Path
import logging
import pandas as pd

from network_idx.constants import (
    FCC_COVERAGE_COUNTY_RESIDUAL_OUTPUTS,
    STATE_USPS_TO_FIPS,
)
from network_idx.config import (
    FEATURES_DIR_FCC_COVERAGE_COUNTY_RESIDUALS,
    PROCESSED_DIR_CENSUS_BAF,
    PROCESSED_DIR_FCC_BROADBAND_COVERAGE)

TECHS = ["copper", "cable", "fiber"]
TIER_METRICS = ["speed_100_20", "less_than_100_20", "more_than_100_20"]
PCT_COLS = [f"{tech}_{metric}" for tech in TECHS for metric in TIER_METRICS]

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)

In [12]:
# Function to load inputs
def _load_inputs(
    state_usps: str,
    fips: str,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Load county parquet, place parquet, and BAF crosswalk for one state.
    """
    county_path = Path("/home/eprashar_solutions_corelogic_com/network-idx/data/processed/fcc/broadband_coverage") / f"fcc_fixed_coverage_county_{state_usps}_{fips}.parquet"
    place_path = Path("/home/eprashar_solutions_corelogic_com/network-idx/data/processed/fcc/broadband_coverage") / f"fcc_fixed_coverage_{state_usps}_{fips}.parquet"
    baf_path = Path("/home/eprashar_solutions_corelogic_com/network-idx/data/processed/census/baf2020") / f"census_baf_{state_usps}_{fips}.parquet"

    for p in (county_path, place_path, baf_path):
        if not p.exists():
            raise FileNotFoundError(f"Missing input: {p}")

    county_df = pd.read_parquet(county_path)
    place_df = pd.read_parquet(place_path)
    baf_df = pd.read_parquet(baf_path, columns=["county_geoid", "place_geoid"])

    return county_df, place_df, baf_df

In [13]:
county_df, place_df, baf_df = _load_inputs("AL", "01")
print(baf_df.head())

  county_geoid place_geoid
0        01001     0162328
1        01001     0162328
2        01001     0162328
3        01001         NaN
4        01001         NaN


In [ ]:
def _build_county_place_map(baf_df: pd.DataFrame) -> pd.DataFrame:
    """
    From the BAF crosswalk, build (county_geoid, place_geoid, place_share).

    place_share = blocks_in_county / total_blocks_in_place.
    For single-county places this is 1.0; for multi-county places it is
    proportional to the block count in each county.
    """
    # Drop blocks not in any place
    cp = baf_df[baf_df["place_geoid"].notna()].copy()

    # Count blocks per (county, place)
    county_place_blocks = (
        cp.groupby(["county_geoid", "place_geoid"])
        .size()
        .rename("blocks_in_county")
        .reset_index()
    )

    # Total blocks per place (across all counties)
    place_total_blocks = (
        cp.groupby("place_geoid")
        .size()
        .rename("total_blocks_in_place")
        .reset_index()
    )

    # Merge and compute share
    merged = county_place_blocks.merge(place_total_blocks, on="place_geoid")
    merged["place_share"] = merged["blocks_in_county"] / merged["total_blocks_in_place"]

    return merged[["county_geoid", "place_geoid", "place_share"]]

In [17]:
county_place_map = _build_county_place_map(baf_df)
print(f"Total rows in county-place map: {len(county_place_map)}")
print("="*80)
print("Sample of places where share is split across multiple counties:")
print(county_place_map.loc[county_place_map["place_share"] < 1.0].sort_values(by="place_geoid"))

Total rows in county-place map: 647
Sample of places where share is split across multiple counties:
    county_geoid place_geoid  place_share
38         01009     0101660     0.188406
238        01055     0101660     0.811594
171        01043     0102116     0.008696
449        01095     0102116     0.991304
322        01073     0102320     0.176471
..           ...         ...          ...
595        01123     0180256     0.029412
266        01057     0182992     0.190476
447        01093     0182992     0.809524
36         01007     0183640     0.666667
606        01125     0183640     0.333333

[103 rows x 3 columns]


In [30]:
# Main function to compute residuals
def compute_residuals(
    county_df: pd.DataFrame,
    place_df: pd.DataFrame,
    county_place_map: pd.DataFrame,
) -> pd.DataFrame:
    """
    For each county, subtract fractionally-allocated place-level absolute
    units from the county total to get residual coverage for non-place blocks.
    """
    # Pre-index place_df by geography_id for fast lookup
    place_indexed = place_df.set_index("geography_id")
    print("Place indexed looks like so:")
    print("="*80)
    print(place_indexed.head(3))

    rows = []

    for _, county_row in county_df.head(3).iterrows():  # TODO: remove slicing to process all counties
        county_geoid = str(county_row["geography_id"])
        county_units = county_row["total_units"]

        # Get all places in this county with their shares
        links = county_place_map[county_place_map["county_geoid"] == county_geoid]
        print(f"Links for county {county_geoid} look like so:")
        print("="*80)
        print(links.head(3))

        # Match to FCC place data (inner join — only places that exist in FCC as well as census mapping
        places_with_share = links[links["place_geoid"].isin(place_indexed.index)].copy()

        if len(places_with_share):
            # Look up total_units for each place
            places_with_share["place_total_units"] = (
                places_with_share["place_geoid"]
                .map(place_indexed["total_units"])
            )

            # Scaled sum by share
            places_units = (places_with_share["place_total_units"] * places_with_share["place_share"]).sum()
            print("places_units calculation:")
            print(places_units)
        else:
            places_units = 0

        residual_units = max(county_units - places_units, 0)

        rec = {
            "county_geoid": county_geoid,
            "state_fips": county_geoid[:2],
            "county_total_units": county_units,
            "places_total_units": round(places_units),
            "residual_units": round(residual_units),
            "place_count": len(places_with_share),
        }

        for col in PCT_COLS:
            if residual_units == 0:
                rec[col] = None
            else:
                county_abs = county_units * county_row.get(col, 0)

                if len(places_with_share) and col in place_indexed.columns:
                    # Σ (place_total_units × place_pct × share)
                    place_pcts = (
                        places_with_share["place_geoid"]
                        .map(place_indexed[col])
                        .fillna(0)
                    )
                    places_abs = (
                        places_with_share["place_total_units"]
                        * place_pcts
                        * places_with_share["place_share"]
                    ).sum()
                else:
                    places_abs = 0

                residual_abs = county_abs - places_abs
                residual_pct = residual_abs / residual_units
                # Cap to [0, 1]
                rec[col] = max(0.0, min(residual_pct, 1.0))

        rows.append(rec)

    result = pd.DataFrame(rows)
    return result[[c for c in FCC_COVERAGE_COUNTY_RESIDUAL_OUTPUTS if c in result.columns]]

In [29]:
# residuals_df = compute_residuals(county_df, place_df, county_place_map)
place_indexed = place_df.set_index("geography_id")
for _, county_row in county_df.head(1).iterrows():
    print(county_row)
    county_geoid = str(county_row["geography_id"])
    links = county_place_map[county_place_map["county_geoid"] == county_geoid]
    print(f"Total places linked to county {county_geoid} are {len(links)} and are:")
    print(links['place_geoid'].tolist())

    # Match to FCC place data (inner join — only places that exist in FCC as well as census mapping
    places_with_share = links[links["place_geoid"].isin(place_indexed.index)].copy()
    #print(f"After matching to FCC place data, total places linked to county {county_geoid} are {len(places_with_share)} and are:")
    #print(places_with_share)

    if len(places_with_share):
            # Look up total_units for each place
            places_with_share["place_total_units"] = (
                places_with_share["place_geoid"]
                .map(place_indexed["total_units"])
            )
            print("Places with share and total units look like so:")
            print("="*80)
            print(places_with_share)
            # Scaled sum by share
            places_units = (places_with_share["place_total_units"] * places_with_share["place_share"]).sum()
            print("places_units calculation:")
            print(places_units)
    else:
        places_units = 0

geography_id                            01001
geography_desc                 Autauga County
geography_desc_full        Autauga County, AL
total_units                             28718
copper_speed_100_20                  0.013162
copper_less_than_100_20              0.373738
copper_more_than_100_20                   0.0
cable_speed_100_20                   0.685946
cable_less_than_100_20               0.685946
cable_more_than_100_20               0.685946
fiber_speed_100_20                   0.556654
fiber_less_than_100_20               0.556654
fiber_more_than_100_20               0.556654
Name: 0, dtype: object
Total places linked to county 01001 are 6 and are:
['0103220', '0106460', '0146600', '0148712', '0160264', '0162328']
Places with share and total units look like so:
  county_geoid place_geoid  place_share  place_total_units
0        01001     0103220     1.000000                478
1        01001     0106460     1.000000                102
2        01001     0146600     1.000